# 第7章　保存・読込・推論・転移学習・次の一歩

学習したモデルを**保存して再利用**し、**学習済みモデルを流用（転移学習）**する方法を学びます。
最後に「よくある落とし穴」チェックリストと、次に進む道を示します。

> **このノートの使い方**
> - 上から順にセルを実行（Colab/Jupyter ともに `Shift + Enter`）。
> - コードは**少し書き換えて壊して直す**のが一番伸びます。各章末に演習があります。
> - GPU は不要な章が多いです。重い章（CNN）では使い方を案内します。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 7-1. モデルの保存と読込（`state_dict` 推奨）

推奨は**重みだけ（`state_dict`）を保存**する方法。モデルの構造はコードで再定義します。

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))

# 保存（重みだけ）
torch.save(model.state_dict(), "model_weights.pth")
print("保存しました")

# 読込：同じ構造を作ってから重みを流し込む
model2 = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))
model2.load_state_dict(torch.load("model_weights.pth"))
model2.eval()    # 推論モードに
print("読み込みOK")

## 7-2. 推論（予測）の定番パターン
本番で予測するときは必ず：**`model.eval()` ＋ `with torch.no_grad():`**。

In [ ]:
model2.eval()
sample = torch.randn(1, 10)         # 1件のダミー入力
with torch.no_grad():
    logits = model2(sample)
    prob = torch.softmax(logits, dim=1)
    pred = logits.argmax(dim=1)
print("確率:", prob, " 予測クラス:", pred.item())

## 7-3. 転移学習（Transfer Learning）— 少ないデータの味方

ImageNet など巨大データで学習済みのモデルを土台にし、**最後の層だけ自分の問題に付け替える**手法。
ゼロから学習するより速く・少データでも高精度。考え方：

1. 学習済みモデルを読み込む（特徴抽出器として優秀）。
2. 前半（特徴抽出部）は**凍結**（`requires_grad=False`）して学習しない。
3. 最終の分類層だけ自分のクラス数に**差し替えて**学習する。

> 下のセルは初回に学習済み重みをダウンロードします（ネット接続が必要、Colob 推奨）。重ければ読むだけでOK。

In [ ]:
from torchvision import models

# 学習済み ResNet18 を読み込み（weights を指定）
net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 1) 特徴抽出部を凍結（勾配を計算しない＝学習で更新しない）
for p in net.parameters():
    p.requires_grad = False

# 2) 最終層 fc を、自分のクラス数（例：花5種）に差し替え
num_classes = 5
net.fc = nn.Linear(net.fc.in_features, num_classes)   # ここだけ requires_grad=True

# 学習対象は差し替えた fc だけ
trainable = [n for n, p in net.named_parameters() if p.requires_grad]
print("学習するパラメータ:", trainable)
# 以降は第6章と同じ学習ループ（optimizer に net.fc.parameters() を渡す）でOK

## 7-4. よくある落とし穴チェックリスト（保存版）

| 症状 | 原因 | 対処 |
|---|---|---|
| 損失が下がらない | `optimizer.zero_grad()` 忘れ／学習率が不適切 | 5ステップを確認、`lr` を 1e-3 付近から調整 |
| `shape` エラー | 次元の不一致 | 各所で `print(x.shape)` |
| 分類で精度が出ない | 出力に自分で `softmax` をかけている | `CrossEntropyLoss` には**生のlogits**を渡す |
| `Expected ... cuda ... cpu` | モデルとデータのデバイス不一致 | 両方 `.to(device)` |
| 評価で結果が不安定 | `model.eval()` / `no_grad()` 忘れ | 推論前に必ず両方 |
| メモリ不足(OOM) | バッチが大きすぎ／勾配を溜めている | `batch_size` を下げる、推論は `no_grad` |
| `loss` が `nan` | 学習率が大きすぎ／入力未正規化 | `lr` を下げる、入力を `Normalize` |
| 過学習（trainだけ高精度） | モデルが複雑／データ少 | データ拡張・`Dropout`・正則化・データ増 |

## 7-5. 次の一歩（ロードマップ）

**深める**
- 公式チュートリアル：https://pytorch.org/tutorials/ （まず "Learn the Basics" → "60 Minute Blitz"）
- 公式ドキュメント：https://pytorch.org/docs/stable/

**分野を選ぶ**
- 画像：データ拡張、ResNet/EfficientNet、`torchvision`、セグメンテーション
- 自然言語・LLM：**Transformer** の仕組み → Hugging Face `transformers`、ファインチューニング
- 音声・時系列：RNN/LSTM、`torchaudio`

**おすすめ教材**
- 書籍『Deep Learning with PyTorch』(Manning, 著者が無料PDF公開)
- 動画：Andrej Karpathy「Neural Networks: Zero to Hero」(YouTube, 自動微分を一から自作)
- 無料書籍：Dive into Deep Learning (https://d2l.ai/) の PyTorch 版

**便利ツール（慣れてきたら）**
- 学習ループの定型を省ける **PyTorch Lightning**
- 実験管理 **TensorBoard** / Weights & Biases
- 学習済みモデルの巨大ハブ **Hugging Face Hub**

## 演習 7
1. 第4章で作ったモデルを保存→別セルで読込→予測まで通してみよう。
2. 第6章の CNN を保存し、新しいランタイムで読み込んで MNIST のテスト精度を再現しよう。
3. （応用）`torchvision.datasets` の小さな画像データで、7-3 の転移学習を最後まで回してみよう。

In [ ]:
# ここに自分のコードを書いて実行してみよう
